# 🌳 Generic (N-ary) Trees — Runnable Notebook

Companion to [`../tutorials/01_Generic_Tree.md`](../tutorials/01_Generic_Tree.md) and the interactive
[`../html/01_generic_tree.html`](../html/01_generic_tree.html).

A **generic tree** lets every node have **any number of children**. Run each cell top to bottom — every idea is
built, demonstrated, and checked with `assert`.

## 1. The node and building a tree
A node = a value + a **list** of children (any number).

In [ ]:
class Node:
    """One node of a generic (N-ary) tree: a value plus a list of child nodes."""
    def __init__(self, val):
        self.val = val
        self.children = []            # any number of children, kept in order

    def __repr__(self):
        return f"Node({self.val!r})"

def add_child(parent, child_val):
    """Create a child node, attach it under `parent`, and return the new node."""
    child = Node(child_val)
    parent.children.append(child)
    return child

# Build this tree:
#            A
#         /  |  \
#        B   C   D
#       / \       \
#      E   F        G
A = Node("A")
B = add_child(A, "B"); C = add_child(A, "C"); D = add_child(A, "D")
E = add_child(B, "E"); F = add_child(B, "F")
G = add_child(D, "G")
print("root:", A, "with", len(A.children), "children")

## 2. Depth-First Search (DFS)
Two natural orders on an N-ary tree: **pre-order** (node *before* its children, top-down) and **post-order**
(node *after* its children, bottom-up).

In [ ]:
def dfs_preorder(node, out=None):
    """Pre-order: record a node BEFORE recursing into its children."""
    if out is None:
        out = []                      # fresh list on the first (outer) call
    if node is None:
        return out
    out.append(node.val)              # visit the node first ...
    for child in node.children:       # ... then each child, left to right
        dfs_preorder(child, out)
    return out

def dfs_postorder(node, out=None):
    """Post-order: record a node AFTER all its children are done."""
    if out is None:
        out = []
    if node is None:
        return out
    for child in node.children:
        dfs_postorder(child, out)
    out.append(node.val)              # visit the node last
    return out

print("pre-order :", dfs_preorder(A))
print("post-order:", dfs_postorder(A))
assert dfs_preorder(A)  == ["A", "B", "E", "F", "C", "D", "G"]
assert dfs_postorder(A) == ["E", "F", "B", "C", "G", "D", "A"]

## 3. Breadth-First Search (BFS)
Visit **level by level** using a **queue** (first-in, first-out).

In [ ]:
from collections import deque

def bfs(root):
    """Breadth-first / level-order: read the tree row by row."""
    order = []
    if root is None:
        return order
    q = deque([root])
    while q:
        node = q.popleft()            # take the oldest waiting node (FIFO)
        order.append(node.val)
        for child in node.children:   # its children wait their turn at the back
            q.append(child)
    return order

print("level-order:", bfs(A))
assert bfs(A) == ["A", "B", "C", "D", "E", "F", "G"]

## 4. Everyday operations = "solve subtrees, then combine"
Notice how `size`, `height`, `count_leaves`, and `find` all share the same shape.

In [ ]:
def size(node):
    """Count nodes = 1 (this node) + the sizes of every subtree."""
    if node is None:
        return 0
    return 1 + sum(size(c) for c in node.children)

def height(node):
    """Longest path DOWN, in edges. Empty tree = -1 so a lone leaf = 0."""
    if node is None:
        return -1
    if not node.children:
        return 0
    return 1 + max(height(c) for c in node.children)

def count_leaves(node):
    """A leaf has no children; otherwise add up the leaves of each subtree."""
    if node is None:
        return 0
    if not node.children:
        return 1
    return sum(count_leaves(c) for c in node.children)

def find(node, target):
    """Does `target` appear anywhere? Check this node, then any subtree."""
    if node is None:
        return False
    if node.val == target:
        return True
    return any(find(c, target) for c in node.children)

print("size       :", size(A))
print("height     :", height(A))
print("leaves     :", count_leaves(A))
print("find 'G'   :", find(A, "G"))
print("find 'Z'   :", find(A, "Z"))
assert size(A) == 7 and height(A) == 2 and count_leaves(A) == 4
assert find(A, "G") and not find(A, "Z")

## 5. Rebuild a tree from flat `{id, parent_id}` rows
How databases store trees. Index rows by id, then link each node to its parent in **one O(n) pass** — even if rows
arrive out of order.

In [ ]:
def build_from_rows(rows):
    """Flat rows -> tree in O(n): index by id, then attach each node to its parent."""
    nodes = {r["id"]: Node(r["id"]) for r in rows}   # step 1: id -> node (a hash map)
    roots = []
    for r in rows:
        pid = r["parent_id"]
        if pid is None:
            roots.append(nodes[r["id"]])             # no parent -> a root
        else:
            nodes[pid].children.append(nodes[r["id"]])   # O(1) parent lookup + link
    return roots

rows = [
    {"id": "A", "parent_id": None},
    {"id": "C", "parent_id": "A"},   # note: C appears before B — order doesn't matter
    {"id": "B", "parent_id": "A"},
    {"id": "E", "parent_id": "B"},
]
roots = build_from_rows(rows)
print("roots        :", roots)
print("A's children :", roots[0].children)
assert len(roots) == 1 and size(roots[0]) == 4

## ✅ Recap
- A generic tree node holds a value + a **list of children**.
- Visit every node with **DFS** (pre/post, recursion) or **BFS** (level-order, a queue) — both `O(n)`.
- Most operations follow **"solve each subtree, then combine"** (sum / max / any).
- Rebuild from flat `{id, parent_id}` rows by indexing in a hash map — `O(n)`.

Next: [`02_Binary_Tree`](../tutorials/02_Binary_Tree.md).